# This loads generated data from **data_generation** without having to run the entire solver again

In [ ]:
import sys
from pathlib import Path

project_path = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(project_path / "src"))

print(
    "Neural network file found:",
    (project_path / "src" / "wing_surrogate" / "neural_network.py").is_file()
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from torch import nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from wing_surrogate.neural_network import build_model as build_nn_model
from wing_surrogate.polynomial_regression import build_model

df = pd.read_csv(project_path / "data" / "wing_data_trial.csv")

print("Number of samples:", len(df))
df.head()

In [ ]:
input_features = [
    "aspect_ratio", "area_m2", "taper_ratio", "tip_twist_deg",
    "alpha_deg", "velocity", "airfoil",
]
output_features = ["CL", "CD_profile"]

# Encode airfoil names so both models can use every DOE input.
X = pd.get_dummies(df[input_features], columns=["airfoil"], dtype=float)
Y = df[output_features]
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.25, random_state=10,
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


# Polynomial Regression

In [ ]:
# Start with degree 1 because our trial dataset is very small
model = build_model(degree=1)

model.fit(X_train, Y_train)

Y_train_pred = model.predict(X_train)
Y_test_pred = model.predict(X_test)

# Errors
for i in range(len(output_features)):

    training_MSE = np.mean(
        (Y_train_pred[:, i] - Y_train.iloc[:, i].to_numpy()) ** 2
    )

    testing_MSE = np.mean(
        (Y_test_pred[:, i] - Y_test.iloc[:, i].to_numpy()) ** 2
    )

    print("Output:", output_features[i])
    print("Training MSE:", training_MSE)
    print("Testing MSE:", testing_MSE)
    print()

# Polynomial Regression Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels = ["Lift Coefficient CL", "Profile Drag Coefficient CD_profile"]
for i, (ax, label) in enumerate(zip(axes, labels)):
    actual = Y_test.iloc[:, i].to_numpy()
    ax.plot(Y_test_pred[:, i], actual, "o")
    ax.plot(
        [actual.min(), actual.max()],
        [actual.min(), actual.max()],
        "r--", label="Perfect Prediction",
    )
    ax.set_xlabel(f"Predicted {label}")
    ax.set_ylabel(f"Actual {label}")
    ax.set_title(f"{label} Predictions")
    ax.grid()
    ax.legend()

plt.tight_layout()
plt.show()


# Neural Network Test

In [ ]:
torch.manual_seed(10)

X_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

Y_scaler = StandardScaler()
Y_train_scaled = Y_scaler.fit_transform(Y_train)

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

Y_train_tensor = torch.tensor(Y_train_scaled, dtype=torch.float32)

In [ ]:
torch.manual_seed(10)

nn_model = build_nn_model(input_size=X_train.shape[1], output_size=len(output_features))

loss_fun = nn.MSELoss()
optimizer = torch.optim.Adam(nn_model.parameters(), lr=0.01)

Epoch = 1000
Train_Losses = []

for epoch in range(Epoch):

    nn_model.train()

    Y_pred = nn_model(X_train_tensor)

    loss = loss_fun(Y_pred, Y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    Train_Losses.append(loss.item())

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}, Training Loss: {loss.item():.6f}")

In [ ]:
nn_model.eval()

with torch.no_grad():
    Y_train_nn_scaled = nn_model(X_train_tensor).numpy()
    Y_test_nn_scaled = nn_model(X_test_tensor).numpy()

Y_train_nn_pred = Y_scaler.inverse_transform(Y_train_nn_scaled)
Y_test_nn_pred = Y_scaler.inverse_transform(Y_test_nn_scaled)

In [ ]:
for i in range(len(output_features)):

    training_MSE = np.mean(
        (Y_train_nn_pred[:, i] - Y_train.iloc[:, i].to_numpy()) ** 2
    )

    testing_MSE = np.mean(
        (Y_test_nn_pred[:, i] - Y_test.iloc[:, i].to_numpy()) ** 2
    )

    print("Output:", output_features[i])
    print("Neural Network Training MSE:", training_MSE)
    print("Neural Network Testing MSE:", testing_MSE)
    print()

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(
    X_test["alpha_deg"], Y_test["CL"],
    "o", label="Solver"
)

plt.plot(
    X_test["alpha_deg"], Y_test_pred[:, 0],
    "s", label="Polynomial Regression"
)

plt.plot(
    X_test["alpha_deg"], Y_test_nn_pred[:, 0],
    "^", label="Neural Network"
)

plt.xlabel("Angle of Attack [deg]")
plt.ylabel("Lift Coefficient CL")
plt.title("Lift Predictions on Held-out DOE Samples")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(
    X_test["alpha_deg"], Y_test["CD_profile"],
    "o", label="Solver"
)

plt.plot(
    X_test["alpha_deg"], Y_test_pred[:, 1],
    "s", label="Polynomial Regression"
)

plt.plot(
    X_test["alpha_deg"], Y_test_nn_pred[:, 1],
    "^", label="Neural Network"
)

plt.xlabel("Angle of Attack [deg]")
plt.ylabel("Profile Drag Coefficient")
plt.title("Profile Drag Predictions on Held-out DOE Samples")
plt.legend()
plt.grid()
plt.show()